# Lab 8-3: Predict Used Car Values Using Decision Tree

In this lab, you will build a Spark ML Pipeline to encode categorical string variables as integers before using them to build a model.

The data used for this exercise concerns various properties of cars, and whether or not these cars were
classified as a good value. The target value to be predicted is `acceptability`, which is a categorical variable representing whether or not a car is considered acceptable for purchase. All other feature variables are also **categorical**.

||Field|Data Type |Description|
|--|--|--|--|
|0|buying|String|Based on selling price|
|1|maint|String|Based on cost to maintain the vehicle|
|2|doors|String|Number of doors|
|3|persons|String|Passenger capacity|
|4|lug_boot|String|Based on luggage boot size|
|5|safety|String|Based on estimated safety of the vehicle|
|6|acceptability|String|Based on overall acceptability of the vehicle|

**Topics**: `VectorAssembler, Pipeline, StringIndexer, DecisionTreeClassifier, MulticlassClassificationEvaluator`

### 1. Download data and upload data files to the lab volume
- First, create the `catalog` (`msbabigdata`), `schema` (`spark`), and `volume` (`lab8_3`) for this lab.
- Download and unzip the data files from [https://idsdl.csom.umn.edu/c/share/sparklab07data.zip](https://idsdl.csom.umn.edu/c/share/sparklab07data.zip). If you still have a copy of the same zip file from Lab 8-1, there is no need to redownload.
- Upload `cars_train.csv` and `cars_test.csv` to the lab volume.

In [ ]:
%sql
create catalog if not exists msbabigdata;
create schema if not exists msbabigdata.spark;
create volume if not exists msbabigdata.spark.lab8_3;

### 2. Load the data files as `train_df` and `test_df` respectively

In the interest of time, we will skip the data exploration steps.

- When you load the data, use the provided `schema_str` below.
- Verify that the data is loaded correctly.

In [ ]:
schema_str="buying string,maint string,doors string,persons string,lug_boot string,safety string,acceptability string"

In [ ]:
path = "/Volumes/msbabigdata/spark/lab8_3"
train_df = spark.read.csv(path+"/cars_train.csv", header=True, schema=schema_str)
test_df = spark.read.csv(path+"/cars_test.csv", header=True, schema=schema_str)

In [ ]:
train_df.limit(5).display()

buying | maint | doors | persons | lug_boot | safety | acceptability
vhigh | vhigh | 2 | 2 | small | med | unacc
vhigh | vhigh | 2 | 2 | small | high | unacc
vhigh | vhigh | 2 | 2 | med | low | unacc
vhigh | vhigh | 2 | 2 | med | med | unacc
vhigh | vhigh | 2 | 2 | med | high | unacc

In [ ]:
test_df.limit(5).display()

buying | maint | doors | persons | lug_boot | safety | acceptability
low | high | 5more | 4 | big | med | acc
low | high | 5more | 4 | big | high | vgood
low | high | 5more | more | small | low | unacc
low | high | 5more | more | small | med | acc
low | high | 5more | more | small | high | acc

### 3. Index String Values

Because string values cannot be directly used in a Spark ML feature vector, we first need to index them using StringIndexer. Create a new [`StringIndexer`](https://spark.apache.org/docs/latest/api/python/pyspark.ml.html#pyspark.ml.feature.StringIndexer) for each of the columns, with the output column name in the form of `[colname]_ix` (for example, `buying` becomes `buying_ix`). Save these seven StringIndexers as `si1`, `si2`, `si3`, and so on.

**Note**: The default sort order of `StringIndexer` is `frequencyDesc`. Others include `frequencyAsc`, `alphabetDesc`, and `alphabetAsc`. See the link above and click "source" to see more details.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
train_df.columns

In [ ]:
si = StringIndexer(
    inputCols = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'acceptability'],
    outputCols=['buying_ix', 'maint_ix', 'doors_ix', 'persons_ix', 'lug_boot_ix', 'safety_ix', 'acceptability_ix']
)
#testing
train_si = si.fit(train_df).transform(train_df)
train_si.limit(5).display()

buying | maint | doors | persons | lug_boot | safety | acceptability | buying_ix | maint_ix | doors_ix | persons_ix | lug_boot_ix | safety_ix | acceptability_ix
vhigh | vhigh | 2 | 2 | small | med | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 1.0 | 1.0 | 0.0
vhigh | vhigh | 2 | 2 | small | high | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 1.0 | 0.0 | 0.0
vhigh | vhigh | 2 | 2 | med | low | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 0.0 | 2.0 | 0.0
vhigh | vhigh | 2 | 2 | med | med | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 0.0 | 1.0 | 0.0
vhigh | vhigh | 2 | 2 | med | high | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 0.0 | 0.0 | 0.0

### 4. Assemble the `features` vector

Spark ML takes input features as vectors. So we need to assemble feature columns into a feature vector column. Create a `VectorAssembler` called `va` to assemble each of the indexed columns **except `acceptability_ix`** into a new column called `features`.

In [ ]:
va = VectorAssembler(
    inputCols= ['buying_ix', 'maint_ix', 'doors_ix', 'persons_ix', 'lug_boot_ix', 'safety_ix'],
    outputCol='features')

    #testing
train_va = va.transform(train_si)
train_va.limit(5).display()

buying | maint | doors | persons | lug_boot | safety | acceptability | buying_ix | maint_ix | doors_ix | persons_ix | lug_boot_ix | safety_ix | acceptability_ix | features
vhigh | vhigh | 2 | 2 | small | med | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 1.0 | 1.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["2.0","0.0","2.0","0.0","1.0","1.0"]}
vhigh | vhigh | 2 | 2 | small | high | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 1.0 | 0.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["2.0","0.0","2.0","0.0","1.0","0.0"]}
vhigh | vhigh | 2 | 2 | med | low | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 0.0 | 2.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["2.0","0.0","2.0","0.0","0.0","2.0"]}
vhigh | vhigh | 2 | 2 | med | med | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 0.0 | 1.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["2.0","0.0","2.0","0.0","0.0","1.0"]}
vhigh | vhigh | 2 | 2 | med | high | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 0.0 | 0.0 | 0.0 | {"type":"0","size":"6","indices":["0","2"],"

### 5. Create a `DecisionTreeClassifier` 

- the label column should be `acceptability_ix` 
- the features column should be `features`.

In [ ]:
dt = DecisionTreeClassifier(labelCol='acceptability_ix', featuresCol='features')

#test
df_dt = dt.fit(train_va).transform(train_va)
df_dt.limit(5).display()

buying | maint | doors | persons | lug_boot | safety | acceptability | buying_ix | maint_ix | doors_ix | persons_ix | lug_boot_ix | safety_ix | acceptability_ix | features | rawPrediction | probability | prediction
vhigh | vhigh | 2 | 2 | small | med | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 1.0 | 1.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["2.0","0.0","2.0","0.0","1.0","1.0"]} | {"type":"1","size":null,"indices":null,"values":["503.0","0.0","0.0","0.0"]} | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","0.0","0.0"]} | 0.0
vhigh | vhigh | 2 | 2 | small | high | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 1.0 | 0.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["2.0","0.0","2.0","0.0","1.0","0.0"]} | {"type":"1","size":null,"indices":null,"values":["503.0","0.0","0.0","0.0"]} | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","0.0","0.0"]} | 0.0
vhigh | vhigh | 2 | 2 | med | low | unacc | 2.0 | 0.0 | 2.0 | 0.0 | 0.0 | 2.0 | 0.0 | {"type":"1","size":null,

### 6. Create an ML Pipeline

Pipeline is a useful tool to streamline the training and application of machine learning models. It combines feature engineering and model fitting steps into a data pipeline. Create a new Spark ML `Pipeline` called `pl`.

- The `steps` should include all of the `StringIndexers`, the `VectorAssembler`, and the `DecisionTreeClassifier`.

In [ ]:
pl = Pipeline(stages=[si, va, dt])

### 7. Train the pipeline
- Train the pipeline on the training data using the `.fit()` method.
- Save the trained pipeline as `plmodel`.

In [ ]:
pl_model = pl.fit(train_df)

### 8. Apply the Pipeline Model to the Test Dataset

- Use the fitted pipeline `plmodel` on `test_df` and save the resulting DataFrame as `predictions`.
- Display sample results, focusing on contrasting predictions with true labels.

In [ ]:
predictions = pl_model.transform(test_df)
predictions.select("acceptability", "acceptability_ix","prediction").limit(15).display()

acceptability | acceptability_ix | prediction
acc | 1.0 | 1.0
vgood | 2.0 | 1.0
unacc | 0.0 | 0.0
acc | 1.0 | 1.0
acc | 1.0 | 1.0
unacc | 0.0 | 0.0
acc | 1.0 | 1.0
vgood | 2.0 | 1.0
unacc | 0.0 | 0.0
acc | 1.0 | 1.0
vgood | 2.0 | 1.0
unacc | 0.0 | 0.0
unacc | 0.0 | 0.0
unacc | 0.0 | 0.0
unacc | 0.0 | 0.0

### 9. Evaluate Model Performance

- Use `MulticlassClassificationEvaluator` to evaluate the `accuracy` of predictions on the test set.

> The available metrics can be found in the [source code for MulticlassClassificationEvaluator](https://spark.apache.org/docs/latest/api/python/_modules/pyspark/ml/evaluation.html#MulticlassClassificationEvaluator).

In [ ]:
# we will use accuracy 
e = MulticlassClassificationEvaluator(labelCol="acceptability_ix", predictionCol="prediction", metricName="accuracy")
e.evaluate(predictions)

This is the end of the Lab